In [1]:
# 06_transfer_noncredit.ipynb
#
# Experiment 6 -- Transfer to non-credit domains.
#
# Purpose:
#   Test whether the two-axis pattern (outcome parity vs proxy leakage) and the
#   governed mitigation transfer beyond credit, by re-running the identical
#   pipeline, changing only the data, on three non-credit benchmarks:
#     Adult/Census income (protected: sex, race),
#     Auto insurance fraud (protected: insured sex),
#     ProPublica COMPAS recidivism (protected: race),
#   with standard ProPublica filtering for COMPAS. 30 seeds, 95% CIs,
#   directional DIR, reconstruction-AUC leakage.
#
# Outputs (PNG+PDF, 600 dpi, grayscale, no captions; paths not printed):
#   results/tables/exp6_transfer.csv
#   results/figures/exp6_<dataset>_<attr>_dir.(png|pdf)

import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
FIG_DIR  = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR  = os.path.join(PROJECT_ROOT, "results", "tables")
for _d in (DATA_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(_d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85"})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

def dir_ratio(y_pred, group):
    approve = (y_pred == 0).astype(int)                 # favorable = good outcome (0)
    r1 = approve[group==1].mean(); r0 = approve[group==0].mean()
    return (r1/r0) if r0 > 0 else np.nan

def eo_gaps(y_true, y_pred, group):
    def rate(ct, mask):
        idx=(y_true==ct)&mask
        return (y_pred[idx]==1).mean() if idx.sum() else np.nan
    a,b = group==1, group==0
    return abs(rate(1,a)-rate(1,b)), abs(rate(0,a)-rate(0,b))

def leakage_auc(Xdf, g):
    Xs = StandardScaler().fit_transform(Xdf.fillna(Xdf.median(numeric_only=True)).values)
    try:
        return cross_val_score(LogisticRegression(max_iter=1000), Xs, g, cv=3,
                               scoring="roc_auc").mean()
    except Exception:
        return np.nan

# ---- Loaders: features_df, protected_series, y[1=unfavorable], group[1=disadvantaged] ----
def load_adult(attr):
    df = pd.read_parquet(os.path.join(DATA_DIR, "adult_income.parquet"))
    y = (df["HIGH_INCOME"] == 0).astype(int).values            # 1 = low income (unfavorable)
    if attr == "sex":  g = (df["sex"].astype(str).str.strip().str.lower()=="female").astype(int).values
    else:              g = (df["race"].astype(str).str.strip()!="White").astype(int).values  # non-white
    drop = ["HIGH_INCOME","income","sex","race","SEX_STD","AGE_YEARS","EDU_STD","VIP","fnlwgt"]
    raw = df.drop(columns=[c for c in drop if c in df.columns])
    cat = [c for c in raw.select_dtypes(include=["object","string"]).columns]
    base = pd.get_dummies(raw, columns=cat).astype(float).reset_index(drop=True)
    prot = pd.Series(g, name="PROTECTED").astype(float)
    return base, prot, y, g

def load_fraud(attr):
    df = pd.read_csv(os.path.join(DATA_DIR, "insurance_fraud.csv"))
    y = (df["fraud_reported"].astype(str).str.upper()=="Y").astype(int).values  # 1 = fraud (unfavorable)
    g = (df["insured_sex"].astype(str).str.upper()=="FEMALE").astype(int).values
    drop = ["fraud_reported","policy_number","policy_bind_date","incident_date",
            "incident_location","insured_zip","insured_sex","auto_model","_c39"]
    raw = df.drop(columns=[c for c in drop if c in df.columns])
    cat = [c for c in raw.select_dtypes(include=["object","string"]).columns]
    base = pd.get_dummies(raw, columns=cat).astype(float).reset_index(drop=True)
    prot = pd.Series(g, name="PROTECTED").astype(float)
    return base, prot, y, g

def load_compas(attr):
    df = pd.read_csv(os.path.join(DATA_DIR, "compas_two_years.csv"))
    # Standard ProPublica filtering
    df = df[(df["days_b_screening_arrest"] <= 30) &
            (df["days_b_screening_arrest"] >= -30) &
            (df["is_recid"] != -1) &
            (df["c_charge_degree"] != "O") &
            (df["score_text"] != "N/A")].copy()
    df = df[df["race"].isin(["African-American","Caucasian"])]
    y = df["two_year_recid"].astype(int).values                 # 1 = recidivism (unfavorable)
    g = (df["race"]=="African-American").astype(int).values
    keep = ["age","priors_count","juv_fel_count","juv_misd_count","juv_other_count",
            "c_charge_degree","sex"]
    raw = df[keep].copy()
    cat = [c for c in raw.select_dtypes(include=["object","string"]).columns]
    base = pd.get_dummies(raw, columns=cat).astype(float).reset_index(drop=True)
    prot = pd.Series(g, name="PROTECTED").astype(float)
    return base, prot, y, g

def top_proxies(base, g, k=4):
    c = base.apply(lambda col: abs(np.corrcoef(col, g)[0,1]) if col.std()>0 else 0.0)
    return c.sort_values(ascending=False).head(k).index.tolist()

def one_run(base, prot, y, g, cond, seed, subsample=None):
    idx = np.arange(len(y))
    if subsample and len(y) > subsample:
        idx = np.random.default_rng(seed).choice(idx, subsample, replace=False)
    b = base.iloc[idx].reset_index(drop=True); pr = prot.iloc[idx].reset_index(drop=True)
    ys, gs = y[idx], g[idx]
    proxies = top_proxies(b, gs, k=4)
    if cond == "with_protected":
        X = pd.concat([b, pr], axis=1); feats = X.columns.tolist()
    elif cond in ("drop_protected","gov_engine"):
        X = b; feats = b.columns.tolist()
    else:
        X = b; feats = [c for c in b.columns if c not in proxies]
    Xtr, Xte, ytr, yte, gtr, gte = train_test_split(
        X[feats], ys, gs, test_size=0.3, random_state=seed, stratify=ys)
    sc = StandardScaler().fit(Xtr)
    m = LogisticRegression(max_iter=2000).fit(sc.transform(Xtr), ytr)
    p = m.predict_proba(sc.transform(Xte))[:,1]
    auc = roc_auc_score(yte, p)
    if cond != "gov_engine":
        yp = (p>=0.5).astype(int)
    else:
        br = ((p>=0.5).astype(int)==0).mean()
        t1 = np.quantile(p[gte==1], br); t0 = np.quantile(p[gte==0], br)
        yp = np.where(gte==1,(p>=t1),(p>=t0)).astype(int)
    dt, dfp = eo_gaps(yte, yp, gte)
    leak = leakage_auc(X[feats], gs) if cond in ("drop_protected","drop_proxies") else np.nan
    return auc, dir_ratio(yp, gte), dt, dfp, leak

DATASETS = [("adult","sex",load_adult,20000),
            ("adult","race",load_adult,20000),
            ("fraud","sex",load_fraud,None),
            ("compas","race",load_compas,None)]
CONDS = ["with_protected","drop_protected","drop_proxies","gov_engine"]
SEEDS = list(range(30))

rows=[]
for dsname, attr, loader, subs in DATASETS:
    base, prot, y, g = loader(attr)
    appr1=(1-y[g==1]).mean(); appr0=(1-y[g==0]).mean()
    base_dir = appr1/appr0 if appr0>0 else np.nan
    for cond in CONDS:
        arr=np.array([one_run(base,prot,y,g,cond,s,subs) for s in SEEDS])
        m=np.nanmean(arr,0); ci=1.96*np.nanstd(arr,0)/np.sqrt(len(SEEDS))
        rows.append({"dataset":dsname,"protected":attr,"condition":cond,
                     "base_data_DIR":round(base_dir,3),
                     "AUC":m[0],"DIR":m[1],"DIR_ci":ci[1],
                     "dTPR":m[2],"dFPR":m[3],"leak_AUC":m[4]})

res=pd.DataFrame(rows)
res.to_csv(os.path.join(TAB_DIR,"exp6_transfer.csv"),index=False)
print(res.drop(columns=["DIR_ci"]).round(3).to_string(index=False))

order=CONDS
for (dsname,attr) in res[["dataset","protected"]].drop_duplicates().itertuples(index=False):
    sub=res[(res.dataset==dsname)&(res.protected==attr)].set_index("condition").loc[order].reset_index()
    fig,ax=plt.subplots(figsize=(6,4.2))
    ax.bar(range(len(sub)),sub["DIR"],yerr=sub["DIR_ci"],capsize=4,
           edgecolor="black",linewidth=1.0,color=GRAYS)
    ax.axhline(0.8,color="black",linestyle="--",linewidth=0.9)
    ax.set_xticks(range(len(sub))); ax.set_xticklabels(sub["condition"],rotation=20,ha="right")
    ax.set_ylabel("Disparate Impact Ratio (DIR)"); ax.set_ylim(0,1.2)
    fig.tight_layout(); save_fig(fig,f"exp6_{dsname}_{attr}_dir"); plt.close(fig)

print("Done: non-credit transfer, 3 domains, 30 seeds.")


dataset protected      condition  base_data_DIR   AUC   DIR  dTPR  dFPR  leak_AUC
  adult       sex with_protected          0.360 0.902 0.286 0.078 0.126       NaN
  adult       sex drop_protected          0.360 0.901 0.302 0.074 0.106     0.929
  adult       sex   drop_proxies          0.360 0.901 0.244 0.086 0.178     0.861
  adult       sex     gov_engine          0.360 0.901 1.000 0.060 0.303       NaN
  adult      race with_protected          0.601 0.901 0.540 0.036 0.089       NaN
  adult      race drop_protected          0.601 0.901 0.595 0.029 0.053     0.747
  adult      race   drop_proxies          0.601 0.901 0.590 0.029 0.054     0.720
  adult      race     gov_engine          0.601 0.901 1.002 0.030 0.168       NaN
  fraud       sex with_protected          1.036 0.848 1.051 0.112 0.030       NaN
  fraud       sex drop_protected          1.036 0.848 1.053 0.113 0.029     0.463
  fraud       sex   drop_proxies          1.036 0.848 1.052 0.100 0.027     0.458
  fraud       se

Done: non-credit transfer, 3 domains, 30 seeds.
